# **Course 2 - Supervised Machine Learning Regression**
**Automatidata project**


## **Data Summary**

This project uses the `2017_Yellow_Taxi_Trip_Data.csv` dataset from the New York City Taxi and Limousine Commission. The file used in this notebook contains **22,699 taxi trips** and **18 original columns**. Each row represents one yellow taxi ride in New York City during 2017.

The dataset includes several groups of variables:

* **Trip information:** pickup time, dropoff time, passenger count, and trip distance
* **Location information:** pickup and dropoff taxi zone IDs
* **Fare and payment information:** fare amount, tips, tolls, taxes, surcharges, and total amount
* **Operational information:** vendor ID, rate code, payment type, and store-and-forward flag

The target variable is **`fare_amount`**, which is the base taxi fare for the ride. The main modeling goal is to predict `fare_amount` using information that would be available before or near the start of a trip, such as passenger count, vendor, pickup/dropoff route information, and rush hour status.

Some columns, such as `tip_amount`, `tolls_amount`, and `total_amount`, are not appropriate predictors for a fare prediction model because they are known after the trip or directly include fare-related information.


## **Objectives**

The objective of this analysis is to build and evaluate regression models that predict taxi `fare_amount`.

The notebook aims to:

* Clean and explore the taxi trip data
* Engineer useful predictors, especially route-based `mean_distance`
* Check important linear regression assumptions
* Build a baseline multiple linear regression model
* Compare regularized linear models: Ridge, Lasso, and ElasticNet
* Identify the best model based on realistic evaluation metrics and data leakage risk


## **Workflow**


### **Part 1. Data Cleaning**
#### **Reading and understanding the data**
Import the packages needed for building linear regression models.


In [ ]:
# Imports
# Packages for numerics + dataframes
import numpy as np
import pandas as pd

# Packages for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Packages for OLS, MLR, and evaluation metrics
import sklearn.metrics as metrics
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


`Pandas` is used to load the NYC TLC dataset.  


In [ ]:
df0 = pd.read_csv("2017_Yellow_Taxi_Trip_Data.csv")

Analyze and discover data. Start with `.shape` and `.info()`.


In [ ]:
# Keep `df0` as the original dataframe and create a copy (df) where changes will go
# Can revert `df` to `df0` if needed down the line
df = df0.copy()

# Display the dataset's shape
print(df.shape)

# Display basic info about the dataset
df.info()

Check for missing data and duplicates using `.isna()` and `.drop_duplicates()`.


In [ ]:
# Check for missing data and duplicates using .isna() and .drop_duplicates()
### YOUR CODE HERE ###

# Check for duplicates
print('Shape of dataframe:', df.shape)
print('Shape of dataframe with duplicates dropped:', df.drop_duplicates().shape)

# Check for missing values in dataframe
print('Total count of missing values:', df.isna().sum().sum())

# Display missing values per column in dataframe
print('Missing values per column:')
df.isna().sum()

**Findings:** 
> There are no duplicates or missing values in the data.


Use `.describe()`.


In [ ]:
# Display descriptive stats about the data
df.describe()

**Findings:** 
> Several patterns stand out in the summary statistics. For example, some variables contain clear outliers, such as `tip_amount` (\$200) and `total_amount` (\$1,200). In addition, several variables, including `mta_tax`, appear to be nearly constant across the dataset, suggesting that they are unlikely to be strong predictors.


##### **Convert pickup & dropoff columns to datetime**


In [ ]:
# Check the format of the data
df['tpep_dropoff_datetime'][0]

In [ ]:
# Convert datetime columns to datetime
# Display data types of `tpep_pickup_datetime`, `tpep_dropoff_datetime`
print('Data type of tpep_pickup_datetime:', df['tpep_pickup_datetime'].dtype)
print('Data type of tpep_dropoff_datetime:', df['tpep_dropoff_datetime'].dtype)

# Convert `tpep_pickup_datetime` to datetime format
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'], format='%m/%d/%Y %I:%M:%S %p')

# Convert `tpep_dropoff_datetime` to datetime format
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'], format='%m/%d/%Y %I:%M:%S %p')

# Display data types of `tpep_pickup_datetime`, `tpep_dropoff_datetime`
print('Data type of tpep_pickup_datetime:', df['tpep_pickup_datetime'].dtype)
print('Data type of tpep_dropoff_datetime:', df['tpep_dropoff_datetime'].dtype)

df.head(3)

#### **Handling the Outliers**

##### **Detection**
Call `df.info()` to inspect the columns and decide which ones to check for outliers.


In [ ]:
df.info()

Some columns in the dataset will not be used in the final model, so we focus outlier checks on the variables that are most relevant to the prediction task. For this project, the most important columns to review are:
* `trip_distance`
* `fare_amount`



##### **Visualization**

Plot a box plot for each feature: `trip_distance` and `fare_amount`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 2))
fig.suptitle('Boxplots for outlier detection')
sns.boxplot(ax=axes[0], x=df['trip_distance'])
sns.boxplot(ax=axes[1], x=df['fare_amount'])
plt.show();

**Findings:** 
>1. The boxplots show outliers in columns: `trip_distance` and `fare_amount`.

>2. The large values in `trip_distance` may still be realistic because taxi trips in New York City can be long. For example, traveling across boroughs can cover many miles. Because of this, the high `trip_distance` values do not need to be changed.

>3. The large values in `fare_amount` are more suspicious. Very high fares or very long trip times may be data errors, so these columns should be checked and possibly adjusted before fitting the model.


##### **`trip_distance` Outliers Imputations**

The summary statistics show that some trips have a `trip_distance` of 0. This could mean the data contains errors, or it could mean that some very short trips were recorded as 0 miles.

To investigate this, sort the unique values in the `trip_distance` column and review the 10 smallest values. This helps determine whether the distances are recorded as rounded numbers or with more precise decimal values.


In [ ]:
# Are trip distances of 0 bad data or very short trips rounded down?
sorted(set(df['trip_distance']))[:10]

The distance values are recorded with a high level of precision, so a value of 0 is unlikely to be caused by rounding. However, a zero-distance trip could still happen if a passenger requested a taxi and then canceled or changed their mind. The next step is to check how often this occurs and decide whether it is common enough to affect the analysis.

Calculate the number of rides where `trip_distance` is 0.


In [ ]:
sum(df['trip_distance']==0)

**Findings:** 
>There are 148 zero-distance trips out of approximately 23,000 rides, which is a relatively small proportion of the data. These values could be replaced with a small value such as 0.01, but doing so is unlikely to meaningfully affect the model. Therefore, the `trip_distance` column will be left unchanged.


##### **`fare_amount` Outliers Imputations**


In [ ]:
df['fare_amount'].describe()

**Findings:**

>The `fare_amount` column has a wide range of values, and some of the extreme values appear unrealistic.

>* **Low values:** Negative fare amounts are not valid and should be corrected. A fare amount of 0 may be possible if a taxi trip was started but then immediately canceled.

>* **High values:** The maximum fare amount in this dataset is nearly \$1,000, which is highly unlikely for a taxi ride. These high values can be capped using a combination of domain knowledge and statistics. The interquartile range (IQR) is \$8. The standard formula, `Q3 + (1.5 * IQR)`, gives a cap of \$26.50, but that is too low for a reasonable maximum taxi fare. Instead, this project uses a larger factor of `6`, which gives a cap of \$62.50.

>Replace any `fare_amount` values below $0 with `0`.


In [ ]:
# Impute values less than $0 with 0
df.loc[df['fare_amount'] < 0, 'fare_amount'] = 0
df['fare_amount'].min()

Now cap the maximum value using `Q3 + (6 * IQR)`. The `outlier_imputer()` function calculates Q3 and the IQR for each selected column, computes this upper limit, and replaces any values above the limit with the calculated maximum.


In [ ]:
def outlier_imputer(column_list, iqr_factor):
    '''
    Impute upper-limit values in specified columns based on their interquartile range.

    Arguments:
        column_list: A list of columns to iterate over
        iqr_factor: A number representing x in the formula:
                    Q3 + (x * IQR). Used to determine maximum threshold,
                    beyond which a point is considered an outlier.

    The IQR is computed for each column in column_list and values exceeding
    the upper threshold for each column are imputed with the upper threshold value.
    '''
    for col in column_list:
        # Reassign minimum to zero
        df.loc[df[col] < 0, col] = 0

        # Calculate upper threshold
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        upper_threshold = q3 + (iqr_factor * iqr)
        print(col)
        print('q3:', q3)
        print('upper_threshold:', upper_threshold)

        # Reassign values > threshold to threshold
        df.loc[df[col] > upper_threshold, col] = upper_threshold
        print(df[col].describe())
        print()

In [ ]:
outlier_imputer(['fare_amount'], 6)

#### **Feature engineering**


##### **Create `mean_distance` column**

Use past trips to estimate the typical distance for each pickup/dropoff pair.

Create a column called `mean_distance`. For each row, this value is calculated by finding all trips with the same pickup and dropoff locations, then taking the average of their `trip_distance` values.

For example, if your data were:

| Trip | Start | End | Distance |
| --: | :---: | :-: | --: |
| 1 | A | B | 1 |
| 2 | C | D | 2 |
| 3 | A | B | 1.5 |
| 4 | D | C | 3 |

The average distance for each route is:
```
A -> B: 1.25 miles
C -> D: 2 miles
D -> C: 3 miles
```

For example, trips 1 and 3 both go from A to B, so their mean distance is `(1 + 1.5) / 2 = 1.25`. Direction matters, so C -> D and D -> C are treated as different routes.

Each row then receives the average distance for its own route:

| Trip | Start | End | Distance | mean_distance |
| --: | :---: | :-: | --: | --: |
| 1 | A | B | 1 | 1.25 |
| 2 | C | D | 2 | 2 |
| 3 | A | B | 1.5 | 1.25 |
| 4 | D | C | 3 | 3 |


We create a helper column called `pickup_dropoff`. This column identifies the pickup/dropoff pair for each row. We convert the pickup and dropoff location IDs to strings and join them with a space between the two values. The space prevents different pairs from being combined incorrectly. For example, pickup/dropoff IDs `12` and `151` should be stored differently from `121` and `51`.

The helper column would look like this:

| Trip | Start | End | pickup_dropoff |
| --: | :---: | :-: | :-- |
| 1 | A | B | 'A B' |
| 2 | C | D | 'C D' |
| 3 | A | B | 'A B' |
| 4 | D | C | 'D C' |


In [ ]:
# Create `pickup_dropoff` column
df['pickup_dropoff'] = df['PULocationID'].astype(str) + ' ' + df['DOLocationID'].astype(str)
df['pickup_dropoff'].head(2)

Next, group the data by the `pickup_dropoff` column and calculate the average `trip_distance` for each pickup/dropoff pair. Store the result in a variable named `grouped`.


In [ ]:
grouped = df.groupby('pickup_dropoff').mean(numeric_only=True)[['trip_distance']]
grouped[:5]

A simpler way to create `mean_distance` is to use `transform('mean')`. This calculates the average `trip_distance` for each `pickup_dropoff` group and returns one value for every row, so the result can be assigned directly to a new column.


In [ ]:
df['mean_distance'] = df.groupby('pickup_dropoff')['trip_distance'].transform('mean')

Confirm that rows with the same pickup and dropoff locations received the same `mean_distance` value.


In [ ]:
# Confirm that it worked
df[(df['PULocationID']==100) & (df['DOLocationID']==231)][['mean_distance']]

##### **Create `day` and `month` columns**

Create two new columns, `day` (name of day) and `month` (name of month) by extracting the relevant information from the `tpep_pickup_datetime` column.


In [ ]:
# Create 'day' col
df['day'] = df['tpep_pickup_datetime'].dt.day_name().str.lower()

# Create 'month' col
df['month'] = df['tpep_pickup_datetime'].dt.strftime('%b').str.lower()

##### **Create `rush_hour` column**

Define rush hour as a ride that meets both conditions:
* The ride happened on a weekday, not Saturday or Sunday.
* The ride started between 06:00 and 10:00, or between 16:00 and 20:00.

Create a binary `rush_hour` column. Use `1` for rides during rush hour and `0` for all other rides.


In [ ]:
# Create binary 'rush_hour' column
pickup_hour = df['tpep_pickup_datetime'].dt.hour
is_weekday = ~df['day'].isin(['saturday', 'sunday'])
is_rush_hour = pickup_hour.between(6, 9) | pickup_hour.between(16, 19)

df['rush_hour'] = (is_weekday & is_rush_hour).astype(int)
df.head()

### **Part 2: Exploratory Data Analysis**

#### **mean_distance vs fare_amount**

Create a scatterplot to visualize the relationship between `mean_distance` and `fare_amount`.


In [ ]:
# Create a scatter plot of mean_distance and fare_amount, with a line of best fit
sns.set_style('whitegrid')
f = plt.figure()
f.set_figwidth(5)
f.set_figheight(5)
sns.regplot(x=df['mean_distance'], y=df['fare_amount'],
            scatter_kws={'alpha':0.5, 's':5},
            line_kws={'color':'red'})
plt.ylim(0, 70)
plt.xlabel('Mean distance')
plt.ylabel('Fare amount')
plt.title('Mean distance vs. fare amount')
plt.show()

The scatter plot shows that `mean_distance` is related to `fare_amount`. It also shows two horizontal lines where many rides have the same fare amount, around \$52 and \$62.50.

The line at \$62.50 comes from the outlier cap applied earlier. Any fare above that limit was replaced with \$62.50.

Next, investigate the other horizontal line by checking how many rides have fare amounts above \$50 and which fare values appear most often.


In [ ]:
df[df['fare_amount'] > 50]['fare_amount'].value_counts().head()

 There are 514 trips whose fares were \$52.

Examine the first 30 of these trips.


In [ ]:
# Set pandas to display all columns
pd.set_option('display.max_columns', None)
df[df['fare_amount']==52].head(30)

**Findings:** 

>Most \$52 fares either start or end at location 132, and all have `RatecodeID` 2.

>`RatecodeID` 2 represents JFK trips. In 2017, taxi rides between JFK Airport and Manhattan had a flat fare of \$52.

>Because these trips have a known flat fare, their predictions can be corrected after the model runs.


#### **Feature Selection**

Select the columns that will be used to train the regression model. Keep the target variable, `fare_amount`, and features that would be available before or at the start of a trip, such as `VendorID`, `passenger_count`, `mean_distance`, and `rush_hour`.

Drop columns that are IDs, helper columns, payment details, or values only known after the trip ends. For example, `tip_amount`, `total_amount`, and `tpep_dropoff_datetime` should not be used as model features because they would not be known when predicting the fare.


In [ ]:
df.info()

In [ ]:
df2 = df.copy()

df2 = df2.drop(['Unnamed: 0', 'tpep_dropoff_datetime', 'tpep_pickup_datetime',
               'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
               'PULocationID', 'DOLocationID', 'payment_type', 'extra',
               'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
               'total_amount', 'pickup_dropoff', 'day', 'month'
               ], axis=1)


df2.info()

#### **Looking for correlations**


Next, create a correlation matrix to see which variables have the strongest linear relationships.


In [ ]:
# Create correlation matrix containing pairwise correlation of columns, using pearson correlation coefficient
df2.corr(method='pearson')

Visualize a correlation heatmap of the data.


In [ ]:
# Create correlation heatmap

plt.figure(figsize=(10,10))
sns.heatmap(df2.corr(method='pearson'), annot=True, cmap='Reds')
plt.title('Correlation heatmap',
          fontsize=18)
plt.show()

**Findings:** 
>`mean_distance` has a strong positive correlation with the target variable, `fare_amount`. This is expected because longer trips usually have higher fares.


### **Part 3. Testing Assumptions for Linear Regression**

Before fitting the final model, check whether the data reasonably satisfies four common linear regression assumptions: linearity, homoscedasticity, normality of residuals, and low multicollinearity.


#### **1. Linearity Assumption**

The linearity assumption means that each numeric predictor should have an approximately linear relationship with the target variable, `fare_amount`. Use `regplot` to compare `fare_amount` with each numeric predictor in the modeling data. `VendorID` is excluded from this check because it is an ID/category, even though it is stored as a number.


In [ ]:
# Check linear relationships between numeric predictors and fare_amount
import math

num_cols = df2.select_dtypes(include='number').columns
features = [col for col in num_cols if col not in ['fare_amount', 'VendorID']]

cols = 3
rows = math.ceil(len(features) / cols)

plt.figure(figsize=(5 * cols, 4 * rows))

for i, col in enumerate(features):
    plt.subplot(rows, cols, i + 1)
    sns.regplot(x=df2[col], y=df2['fare_amount'],
                scatter_kws={'alpha': 0.4, 's': 5},
                line_kws={'color': 'red'})
    plt.title(f'{col} vs. fare_amount')
    plt.xlabel(col)
    plt.ylabel('Fare amount')

plt.tight_layout()
plt.show()

**Findings:** 
>`mean_distance` shows the clearest positive linear relationship with `fare_amount`, which is expected because longer trips usually cost more. `passenger_count` and `rush_hour` are discrete variables, so their plots appear in vertical bands and show weaker linear patterns. Overall, the linearity assumption appears most reasonable for `mean_distance`.


#### **2. Homoscedasticity**

Homoscedasticity means that the prediction errors should have a similar spread across the range of the predictor. Here, use a residual plot for `mean_distance` and `fare_amount` to check whether the residuals are scattered evenly around 0. 

A residual plot calculates the residual for each data point by subtracting the predicted fare_amount value from the actual value:

$\text{Residual} = y - \hat{y}$ 

First, a regression line is fitted to the data. Then, the model predicts a fare_amount value ($\hat{y}$) for each mean_distance ($x$)-value. The residual is the vertical distance between the actual fare_amount value ($y$) and the predicted fare_amount value on the regression line.


In [ ]:
# Check whether residual spread is roughly constant
plt.figure(figsize=(6, 5))
sns.residplot(x=df2['mean_distance'], y=df2['fare_amount'],
              scatter_kws={'alpha': 0.4, 's': 10},
              line_kws={'color': 'red'})
plt.xlabel('Mean distance')
plt.ylabel('Residuals')
plt.title('Homoscedasticity check')
plt.show()

**Findings:** 
>The residuals are generally centered around 0, but the spread is not completely constant. The residuals appear wider for very short and longer mean distances, which suggests some heteroscedasticity. The assumption is not perfectly met, but the pattern does not appear extreme.


#### **3. Normality**

The normality assumption means that the model's residuals should be approximately normally distributed. In this section, calculate residuals from the selected modeling features, then use a histogram and Q-Q plot to compare their shape to a normal distribution.


In [ ]:
# Check whether residuals are approximately normal
from scipy import stats

X_normality = df2.drop(columns=['fare_amount']).copy()
y_normality = df2['fare_amount']

# Treat VendorID as categorical, then dummy encode it
X_normality['VendorID'] = X_normality['VendorID'].astype(str)
X_normality = pd.get_dummies(X_normality, drop_first=True)

normality_model = LinearRegression().fit(X_normality, y_normality)
normality_residuals = y_normality - normality_model.predict(X_normality)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(normality_residuals, bins=40, kde=True, ax=axes[0])
axes[0].set_title('Residual distribution')
axes[0].set_xlabel('Residual')

stats.probplot(normality_residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q plot of residuals')

plt.tight_layout()
plt.show()

**Findings:** The residuals are centered close to 0, but they are not perfectly normal. The histogram is right-skewed, and the Q-Q plot shows departures from the diagonal line, especially in the tails. This suggests that the normality assumption is only partially met.


#### **4. Multicollinearity**

Multicollinearity means that predictors are highly related to each other. Check this with a correlation heatmap and variance inflation factor (VIF) values. VIF is calculated by predicting each feature from the other features; higher VIF values mean that a feature is more strongly explained by the others.


In [ ]:
# Check predictor correlations
X_vif = df2.drop(columns=['fare_amount']).copy()
X_vif['VendorID'] = X_vif['VendorID'].astype(str)
X_vif = pd.get_dummies(X_vif, drop_first=True)

plt.figure(figsize=(10, 10))
sns.heatmap(X_vif.corr(), annot=True, cmap='Reds', fmt='.2f')
plt.title('Predictor correlation heatmap')
plt.show()

In [ ]:
# Calculate VIF for each predictor
vif_values = []

for feature in X_vif.columns:
    X_other = X_vif.drop(columns=[feature])
    y_feature = X_vif[feature]

    r_squared = LinearRegression().fit(X_other, y_feature).score(X_other, y_feature)
    vif = np.inf if r_squared >= 1 else 1 / (1 - r_squared)
    vif_values.append({'feature': feature, 'VIF': vif})

vif_df = pd.DataFrame(vif_values).sort_values('VIF', ascending=False)
vif_df

**Findings:** The predictors do not show strong multicollinearity. The heatmap shows low correlations between most predictors, and all VIF values are close to 1. This suggests that the model features are not strongly explained by one another.


### **Part 4. Data Preparation**

Prepare the selected modeling data for linear regression. The target is `fare_amount`, and the predictors are the remaining columns in `df2`: `VendorID`, `passenger_count`, `mean_distance`, and `rush_hour`.


In [ ]:
df2.info()


#### **Define Features and Target**

Separate the data into `X` and `y`. `X` contains the predictor variables used by the model, and `y` contains the target variable, `fare_amount`.


In [ ]:
# Remove the target column from the features
X = df2.drop(columns=['fare_amount'])

# Set y variable
y = df2[['fare_amount']]

# Display first few rows
X.head()


#### **Encode Categorical Features**

Convert `VendorID` to a categorical variable and dummy encode it. This allows the linear regression model to use vendor information without treating the vendor ID as a continuous number.


In [ ]:
# Convert VendorID to string
X['VendorID'] = X['VendorID'].astype(str)

# Get dummies
X = pd.get_dummies(X, drop_first=True)
X.head()


#### **Train Test Split**

Split the data into training and test sets. The model will learn from the training set, and the test set will be used later to evaluate how well the model performs on unseen data. Use 20% of the data for testing and set `random_state=0` so the split is reproducible.


In [ ]:
# Create training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


#### **Standardize the data**

Standardize the predictor variables so they are on a similar scale. Fit the `StandardScaler` on `X_train` only, then use it to transform `X_train`. The same fitted scaler will be used later to transform `X_test`.


In [ ]:
# Standardize the X variables
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
print('X_train scaled:', X_train_scaled)


### **Part 5. Linear Regression**

#### **Fit Linear Regression Model**

Instantiate a linear regression model and fit it to the scaled training data. The model will learn the relationship between the selected predictors and `fare_amount`.


In [ ]:
# Fit the model to the training data
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)


#### **Evaluate Model Performance**

Evaluate the model on both the training and test data. Comparing these results helps determine whether the model generalizes well to unseen data.


##### **Training Set Performance**

First, evaluate the model on the training data. Use R-squared to measure how much variation in `fare_amount` is explained by the model, and use MAE, MSE, and RMSE to measure prediction error.


In [ ]:
# Evaluate the model performance on the training data
r_sq = lr.score(X_train_scaled, y_train)
print('Coefficient of determination:', r_sq)
y_pred_train = lr.predict(X_train_scaled)
print('R^2:', r2_score(y_train, y_pred_train))
print('MAE:', mean_absolute_error(y_train, y_pred_train))
print('MSE:', mean_squared_error(y_train, y_pred_train))
print('RMSE:',np.sqrt(mean_squared_error(y_train, y_pred_train)))


##### **Test Set Performance**

Next, evaluate the model on the test data. Transform `X_test` using the scaler that was fit on `X_train`; do not refit the scaler on the test data.


In [ ]:
# Scale the X_test data
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Evaluate the model performance on the testing data
r_sq_test = lr.score(X_test_scaled, y_test)
print('Coefficient of determination:', r_sq_test)
y_pred_test = lr.predict(X_test_scaled)
print('R^2:', r2_score(y_test, y_pred_test))
print('MAE:', mean_absolute_error(y_test,y_pred_test))
print('MSE:', mean_squared_error(y_test, y_pred_test))
print('RMSE:',np.sqrt(mean_squared_error(y_test, y_pred_test)))


**Findings:** The model performs well on both the training and test sets. The test R-squared is about 0.85, meaning the model explains roughly 85% of the variation in `fare_amount` for unseen data. The training and test scores are close, which suggests that the model is not strongly overfit. The MAE is about $2.45 on the test set, so the model's average prediction error is around two to three dollars.


#### **Create Results DataFrame**

Create a `results` dataframe for the test set. It contains the actual fare, the predicted fare, and the residual for each test observation.


In [ ]:
# Create a `results` dataframe
results = pd.DataFrame(data={'actual': y_test['fare_amount'],
                             'predicted': y_pred_test.ravel()})
results['residual'] = results['actual'] - results['predicted']
results.head()


#### **Visualize Model Results**


Plot actual fares against predicted fares. Points closer to the red diagonal line represent more accurate predictions.


In [ ]:
# Create a scatterplot to visualize `predicted` over `actual`
fig, ax = plt.subplots(figsize=(6, 6))
sns.set(style='whitegrid')
sns.scatterplot(x='actual',
                y='predicted',
                data=results,
                s=20,
                alpha=0.5,
                ax=ax
)
# Draw an x=y line to show what the results would be if the model were perfect
plt.plot([0,60], [0,60], c='red', linewidth=2)
plt.title('Actual vs. predicted');

Plot the distribution of residuals to see whether prediction errors are centered near 0.


In [ ]:
# Visualize the distribution of the `residuals`
sns.histplot(results['residual'], bins=np.arange(-15,15.5,0.5))
plt.title('Distribution of the residuals')
plt.xlabel('residual value')
plt.ylabel('count');

In [ ]:
results['residual'].mean()

**Findings:** The residuals are centered close to 0, which suggests that the model is not consistently overpredicting or underpredicting fares. However, the distribution is right-skewed, meaning some rides have larger positive errors than others.


Plot residuals against predicted fares to check whether the errors show any clear pattern.


In [ ]:
# Create a scatterplot of `residuals` over `predicted`

sns.scatterplot(x='predicted', y='residual', data=results)
plt.axhline(0, c='red')
plt.title('Scatterplot of residuals over predicted values')
plt.xlabel('predicted value')
plt.ylabel('residual value')
plt.show()

**Findings:** Most residuals are centered around 0, but there are visible diagonal bands. These bands are likely related to repeated fare values, including the $62.50 outlier cap and the $52 JFK flat fare.


#### **Interpret Model Coefficients**

Use the model coefficients to understand how each feature contributes to the fare prediction. Because the predictors were standardized, the coefficients are measured in standard deviation units rather than the original units.


In [ ]:
# Get model coefficients
coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lr.coef_.ravel()
})

coefficients['absolute_coefficient'] = coefficients['coefficient'].abs()
coefficients.sort_values('absolute_coefficient', ascending=False)


**Findings:** 
> The largest coefficient belongs to `mean_distance`, so this feature has the strongest influence on the model's fare predictions. Because the data was standardized, this coefficient means that a one-standard-deviation increase in `mean_distance` is associated with an increase in predicted fare, holding the other features constant.

To make the interpretation easier, convert the coefficient back to the original unit of miles by dividing the `mean_distance` coefficient by the standard deviation of `mean_distance` in the training data.


In [ ]:
# Convert the standardized mean_distance coefficient back to dollars per mile
mean_distance_coef = coefficients.loc[
    coefficients['feature'] == 'mean_distance', 'coefficient'
].iloc[0]

mean_distance_std = X_train['mean_distance'].std()
fare_increase_per_mile = mean_distance_coef / mean_distance_std

print('Mean distance standard deviation:', mean_distance_std)
print('Mean distance coefficient:', mean_distance_coef)
print('Estimated fare increase per mile:', fare_increase_per_mile)

**Findings:** 
> After converting back to miles, each additional mile in `mean_distance` is associated with an estimated fare increase of about $2.68, holding the other features constant.


### **Part 6. Regularization**

Regularization helps reduce overfitting by adding a penalty to large model coefficients. This can make a linear regression model more stable on new data.

This section compares three regularized linear models:

* **Ridge regression:** Shrinks coefficients toward zero, but usually keeps all features in the model.
* **Lasso regression:** Can shrink some coefficients exactly to zero, so it can help with feature selection.
* **ElasticNet regression:** Combines Ridge and Lasso regularization.

One key concern is data leakage from `mean_distance`. The first linear regression model used a `mean_distance` feature calculated from the full dataset, so its performance is likely too optimistic. The regularized models avoid this leakage with a pipeline, so the validation and test sets use the route-mean lookup table learned from the training data.

A reusable function, `plot_regularization_path()`, is used to create the coefficient path and R-squared plots for all three models.


#### **Ridge Regression**

Ridge regression adds an L2 penalty to linear regression. This penalty shrinks large coefficients and can help the model generalize better.

This Ridge workflow uses:

* `RouteMeanDistanceTransformer` to create `mean_distance` without leakage
* `ColumnTransformer` to scale numeric features and encode `VendorID`
* `Pipeline` to keep preprocessing and modeling together
* shuffled 5-fold `KFold` cross-validation
* `GridSearchCV` to choose the best Ridge `alpha`

The `alpha` parameter controls the strength of regularization. Larger `alpha` values mean stronger coefficient shrinkage.


##### **Create a Leakage-Safe Route Transformer**

This transformer creates `mean_distance` inside the pipeline without using validation or test data for the mean calculation. This prevents data leakage.

In `fit()`, it learns route mean distances from the training data only.

In `transform()`, it applies those learned route means to the current data. For validation or test data, it does not recalculate route means; it only looks up the route means learned from the training data.

The rule is:

* If the route was seen in training, use that route’s average training distance.
* If the route was not seen in training, use the overall average training distance.




In [ ]:
class RouteMeanDistanceTransformer(BaseEstimator, TransformerMixin):
    """Create mean_distance from training-fold route history only."""

    def fit(self, X, y=None):
        X = X.copy()
        routes = X['PULocationID'].astype(str) + ' ' + X['DOLocationID'].astype(str)
        self.route_mean_distance_ = X.assign(route=routes).groupby('route')['trip_distance'].mean()
        self.global_mean_distance_ = X['trip_distance'].mean()
        return self

    def transform(self, X):
        X = X.copy()
        routes = X['PULocationID'].astype(str) + ' ' + X['DOLocationID'].astype(str)

        X_transformed = pd.DataFrame(index=X.index)
        X_transformed['VendorID'] = X['VendorID'].astype(str)
        X_transformed['passenger_count'] = X['passenger_count']
        X_transformed['mean_distance'] = routes.map(self.route_mean_distance_).fillna(
            self.global_mean_distance_
        )
        X_transformed['rush_hour'] = X['rush_hour']
        return X_transformed


##### **Prepare Raw Features for the Pipeline**

Use the raw pickup/dropoff columns and `trip_distance` here because the transformer needs them to learn route means from the training data.

These raw columns are used inside the pipeline to create `mean_distance`. The validation and test sets use the route means learned from training; they do not create separate route means from their own `trip_distance` values.


In [ ]:
# Use raw route columns so the pipeline can create mean_distance without leakage
ridge_features = ['VendorID', 'passenger_count', 'PULocationID', 'DOLocationID',
                  'trip_distance', 'rush_hour']
X_ridge = df[ridge_features].copy()
y_ridge = df['fare_amount']

X_ridge_train, X_ridge_test, y_ridge_train, y_ridge_test = train_test_split(
    X_ridge, y_ridge, test_size=0.2, random_state=0
)


##### **Set Up Preprocessing and Cross-Validation**

The preprocessor scales the numeric features and one-hot encodes `VendorID`. The same preprocessor is reused for Ridge, Lasso, ElasticNet, and the regularization path plots.


In [ ]:
ridge_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['passenger_count', 'mean_distance', 'rush_hour']),
        ('vendor', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['VendorID'])
    ]
)

cv = KFold(n_splits=5, shuffle=True, random_state=0)


##### **Tune Ridge with GridSearchCV**

Grid search tests several Ridge `alpha` values. The best value is the one with the lowest cross-validated RMSE.


In [ ]:
ridge_pipe = Pipeline(
    steps=[
        ('route_mean_distance', RouteMeanDistanceTransformer()),
        ('preprocess', ridge_preprocessor),
        ('ridge', Ridge())
    ]
)

param_grid = {
    'ridge__alpha': np.logspace(-3, 3, 13)
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

ridge_grid.fit(X_ridge_train, y_ridge_train)

ridge_pred = ridge_grid.predict(X_ridge_test)

print('Best alpha:', ridge_grid.best_params_['ridge__alpha'])
print('Best CV RMSE:', -ridge_grid.best_score_)
print('Test R^2:', r2_score(y_ridge_test, ridge_pred))
print('Test MAE:', mean_absolute_error(y_ridge_test, ridge_pred))
print('Test RMSE:', np.sqrt(mean_squared_error(y_ridge_test, ridge_pred)))


##### **Create a Reusable Regularization Path Plot Function**

This helper function is used for Ridge, Lasso, and ElasticNet. For each `alpha`, it fits the full leakage-safe pipeline, stores the coefficients, calculates test R-squared, and makes two plots.


In [ ]:
def plot_regularization_path(model_name, model_factory, alphas, estimator_step_name,
                             X_train, y_train, X_test, y_test):
    """Plot coefficient paths and test R-squared values across alpha values."""
    coefs = []
    r2_scores = []

    for alpha in alphas:
        path_pipe = Pipeline(
            steps=[
                ('route_mean_distance', RouteMeanDistanceTransformer()),
                ('preprocess', ridge_preprocessor),
                (estimator_step_name, model_factory(alpha))
            ]
        )

        path_pipe.fit(X_train, y_train)
        estimator = path_pipe.named_steps[estimator_step_name]

        coefs.append(estimator.coef_)
        r2_scores.append(path_pipe.score(X_test, y_test))

    coefs = np.array(coefs)
    feature_names = path_pipe.named_steps['preprocess'].get_feature_names_out()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(alphas, coefs)
    axes[0].set_xscale('log')
    axes[0].set_xlabel('alpha')
    axes[0].set_ylabel('coefficient')
    axes[0].set_title(f'{model_name} coefficients as alpha changes')
    axes[0].legend(feature_names, fontsize=8)

    axes[1].plot(alphas, r2_scores, marker='o')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('alpha')
    axes[1].set_ylabel('$R^2$')
    axes[1].set_title(f'{model_name} $R^2$ as alpha changes')

    plt.tight_layout()
    plt.show()

    return pd.DataFrame({'alpha': alphas, 'r2_score': r2_scores})


##### **Ridge Regularization Path Plots**

The next two plots show how Ridge changes as `alpha` changes.

* The coefficient plot shows how each feature coefficient changes.
* The R-squared plot shows how test performance changes.

Each point is fit with the full leakage-safe pipeline.


In [ ]:
ridge_path_scores = plot_regularization_path(
    model_name='Ridge',
    model_factory=lambda alpha: Ridge(alpha=alpha),
    alphas=param_grid['ridge__alpha'],
    estimator_step_name='ridge',
    X_train=X_ridge_train,
    y_train=y_ridge_train,
    X_test=X_ridge_test,
    y_test=y_ridge_test
)

ridge_path_scores


**Ridge Findings:**

The best Ridge `alpha` was approximately `316.23`. The cross-validation RMSE was about `6.49`, and the test RMSE was about `6.18`.

The test R-squared was about `0.65`, so the leakage-safe Ridge model explains about 65% of the variation in `fare_amount` on the test set.

This score is lower than the earlier linear regression result because the earlier model used a `mean_distance` feature calculated from the full dataset. The Ridge result is more realistic because it avoids that leakage.

In the path plots, Ridge shrinks coefficients gradually as `alpha` increases. Ridge usually keeps all features in the model.


#### **Lasso Regression**

Lasso regression adds an L1 penalty to linear regression. Like Ridge, it shrinks coefficients. Unlike Ridge, Lasso can shrink some coefficients exactly to zero.

This makes Lasso useful for feature selection. If a coefficient becomes zero, that feature is not contributing to the model at that level of regularization.

This Lasso workflow uses the same leakage-safe pipeline and shuffled cross-validation as Ridge. `GridSearchCV` tunes the Lasso `alpha`.


In [ ]:
lasso_pipe = Pipeline(
    steps=[
        ('route_mean_distance', RouteMeanDistanceTransformer()),
        ('preprocess', ridge_preprocessor),
        ('lasso', Lasso(max_iter=20000, random_state=0))
    ]
)

lasso_param_grid = {
    'lasso__alpha': np.logspace(-3, 1, 9)
}

lasso_grid = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

lasso_grid.fit(X_ridge_train, y_ridge_train)

lasso_pred = lasso_grid.predict(X_ridge_test)

print('Best alpha:', lasso_grid.best_params_['lasso__alpha'])
print('Best CV RMSE:', -lasso_grid.best_score_)
print('Test R^2:', r2_score(y_ridge_test, lasso_pred))
print('Test MAE:', mean_absolute_error(y_ridge_test, lasso_pred))
print('Test RMSE:', np.sqrt(mean_squared_error(y_ridge_test, lasso_pred)))

lasso_feature_names = lasso_grid.best_estimator_.named_steps['preprocess'].get_feature_names_out()
lasso_coefficients = pd.DataFrame({
    'feature': lasso_feature_names,
    'coefficient': lasso_grid.best_estimator_.named_steps['lasso'].coef_
})

lasso_coefficients['absolute_coefficient'] = lasso_coefficients['coefficient'].abs()
lasso_coefficients.sort_values('absolute_coefficient', ascending=False)


##### **Lasso Regularization Path Plots**

The next two plots show how Lasso changes as `alpha` changes.

Lasso coefficients may drop to zero as regularization gets stronger. This makes the coefficient path useful for seeing which features remain important.

The plots use the same reusable `plot_regularization_path()` function.


In [ ]:
lasso_path_scores = plot_regularization_path(
    model_name='Lasso',
    model_factory=lambda alpha: Lasso(alpha=alpha, max_iter=20000, random_state=0),
    alphas=lasso_param_grid['lasso__alpha'],
    estimator_step_name='lasso',
    X_train=X_ridge_train,
    y_train=y_ridge_train,
    X_test=X_ridge_test,
    y_test=y_ridge_test
)

lasso_path_scores


**Lasso Findings:**

The best Lasso `alpha` was `0.10`. The cross-validation RMSE was about `6.49`, and the test RMSE was about `6.18`.

The test R-squared was about `0.65`, which is very close to the Ridge result. Lasso did not greatly improve performance, but it is still useful for checking which coefficients are strongest and whether any are pushed toward zero.

Lasso had the slightly lowest test RMSE in this run, but the difference from Ridge and ElasticNet is very small.


#### **ElasticNet Regression**

ElasticNet combines Ridge and Lasso regularization. It has two main parameters:

* `alpha`: controls the overall strength of regularization
* `l1_ratio`: controls the mix of Lasso and Ridge penalties

When `l1_ratio` is close to `1`, ElasticNet behaves more like Lasso. When `l1_ratio` is close to `0`, it behaves more like Ridge.

This ElasticNet workflow uses the same leakage-safe pipeline and shuffled cross-validation. `GridSearchCV` tunes both `alpha` and `l1_ratio`.


In [ ]:
elasticnet_pipe = Pipeline(
    steps=[
        ('route_mean_distance', RouteMeanDistanceTransformer()),
        ('preprocess', ridge_preprocessor),
        ('elasticnet', ElasticNet(max_iter=20000, random_state=0))
    ]
)

elasticnet_param_grid = {
    'elasticnet__alpha': np.logspace(-3, 1, 9),
    'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elasticnet_grid = GridSearchCV(
    estimator=elasticnet_pipe,
    param_grid=elasticnet_param_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

elasticnet_grid.fit(X_ridge_train, y_ridge_train)

elasticnet_pred = elasticnet_grid.predict(X_ridge_test)

print('Best parameters:', elasticnet_grid.best_params_)
print('Best CV RMSE:', -elasticnet_grid.best_score_)
print('Test R^2:', r2_score(y_ridge_test, elasticnet_pred))
print('Test MAE:', mean_absolute_error(y_ridge_test, elasticnet_pred))
print('Test RMSE:', np.sqrt(mean_squared_error(y_ridge_test, elasticnet_pred)))

elasticnet_feature_names = elasticnet_grid.best_estimator_.named_steps['preprocess'].get_feature_names_out()
elasticnet_coefficients = pd.DataFrame({
    'feature': elasticnet_feature_names,
    'coefficient': elasticnet_grid.best_estimator_.named_steps['elasticnet'].coef_
})

elasticnet_coefficients['absolute_coefficient'] = elasticnet_coefficients['coefficient'].abs()
elasticnet_coefficients.sort_values('absolute_coefficient', ascending=False)


##### **ElasticNet Regularization Path Plots**

ElasticNet has both `alpha` and `l1_ratio`. To keep the plots simple, these plots change `alpha` while holding `l1_ratio` at the best value found by grid search.

The plots use the same reusable `plot_regularization_path()` function as Ridge and Lasso.


In [ ]:
best_l1_ratio = elasticnet_grid.best_params_['elasticnet__l1_ratio']

elasticnet_path_scores = plot_regularization_path(
    model_name=f'ElasticNet, l1_ratio={best_l1_ratio}',
    model_factory=lambda alpha: ElasticNet(alpha=alpha, l1_ratio=best_l1_ratio,
                                           max_iter=20000, random_state=0),
    alphas=elasticnet_param_grid['elasticnet__alpha'],
    estimator_step_name='elasticnet',
    X_train=X_ridge_train,
    y_train=y_ridge_train,
    X_test=X_ridge_test,
    y_test=y_ridge_test
)

elasticnet_path_scores


**ElasticNet Findings:**

The best ElasticNet parameters were `alpha = 0.10` and `l1_ratio = 0.90`. Because `l1_ratio` is close to `1`, this ElasticNet model behaves more like Lasso than Ridge.

The cross-validation RMSE was about `6.49`, the test RMSE was about `6.18`, and the test R-squared was about `0.65`.

Ridge, Lasso, and ElasticNet perform almost the same in this notebook. Using the unrounded test RMSE values, Lasso was slightly lowest, followed by ElasticNet, then Ridge. However, the differences are very small, so there is no strong practical difference between the three regularized models.

The most important improvement is the leakage-safe pipeline for `mean_distance`, not the small difference between Ridge, Lasso, and ElasticNet.


### **Part 7. Conclusion**

#### **Model Comparison**

This notebook compares a baseline multiple linear regression model with three regularized linear regression models.

* **Baseline multiple linear regression:** This model had the highest test R-squared, about `0.85`, and a test MAE of about `$2.45`. However, this result is likely too optimistic because the `mean_distance` feature was calculated from the full dataset before the train/test split.
* **Ridge regression:** The leakage-safe Ridge model had a test R-squared of about `0.649` and a test RMSE of about `6.179`.
* **Lasso regression:** The leakage-safe Lasso model had a test R-squared of about `0.649` and a test RMSE of about `6.175`.
* **ElasticNet regression:** The leakage-safe ElasticNet model had a test R-squared of about `0.649` and a test RMSE of about `6.178`.

The baseline linear regression model appears stronger by raw metrics, but the regularized pipeline models are more trustworthy because they prevent `mean_distance` data leakage.

Among the leakage-safe models, **Lasso is the slight numerical winner** because it has the lowest unrounded test RMSE. However, the difference is very small, so Ridge, Lasso, and ElasticNet should be treated as performing almost the same in this notebook. Lasso is still a reasonable final choice because it performs marginally best and can shrink less useful coefficients toward zero.

#### **Key Findings**

`mean_distance` is the most important predictor of `fare_amount`. This makes sense because longer routes usually have higher taxi fares.

The `$52` fare pattern is related to `RatecodeID 2`, which represents JFK flat-fare trips. These trips follow a known fare rule, so they may be better handled separately instead of forcing the regression model to learn them.

The baseline model's high performance should be interpreted carefully because of leakage in the original `mean_distance` feature. The regularized models give a more realistic estimate of performance.

Ridge, Lasso, and ElasticNet performed almost the same. This suggests that, with the current feature set, regularization choice matters less than preventing leakage and using appropriate predictors.

#### **Limitations and Next Steps**

The earlier baseline model has data leakage because `mean_distance` was calculated before the train/test split. Future baseline models should also use a pipeline, like the regularized models.

The model uses only a small number of predictors. Future work could add more valid pre-trip features, such as pickup hour, day of week, borough information, airport indicators, or route-level historical patterns calculated safely within cross-validation.

`RatecodeID 2` JFK flat-fare trips should be separated and handled with a rule-based approach. The regression model can then focus on trips where the fare actually needs to be predicted.

Some linear regression assumptions were only partially met. The residuals were not perfectly normal and showed some patterning, so future work could compare linear models with other approaches, such as tree-based models.

A production workflow should use a single end-to-end modeling pipeline for feature engineering, preprocessing, cross-validation, model tuning, and prediction.
